# Gate 2 — Identity Preservation Calibration

**Purpose:** Verify that FACodec reconstruction preserves speaker identity.

**Method:** Three-reference calibration with ECAPA-TDNN speaker embeddings:
1. SAME-SPEAKER reference  — within-speaker cosine similarity
2. DIFFERENT-SPEAKER ref   — impostor cosine similarity floor
3. RECONSTRUCTION ref      — source vs FACodec reconstruction similarity

**Pass criteria:**
- shift_over_span < 0.25  (reconstruction damage < 25% of natural speaker variation)
- preservation_ratio > 0.85  (reconstruction retains > 85% of same-speaker identity)

**Stop condition:** If either criterion fails for native or Indian corpus → FAIL.

In [ ]:
# 1. Mount Google Drive
from google.colab import drive
drive.mount('/content/drive')
print("Drive mounted at /content/drive")

In [ ]:
# 2. Setup paths and load Gate 1 artifacts
import os, sys, json, time, subprocess, types, warnings, shutil
from pathlib import Path

warnings.simplefilter("ignore")

ACCENTEDGE_DIR = "/content/accentedge"
FA_CODEC_DIR   = "/content/FAcodec"
GATE1_DIR      = "/content/gate1_artifacts"
GATE2_DIR      = "/content/gate2_artifacts"
DRIVE_BASE     = "/content/drive/MyDrive/accentedge/runs"

def run(cmd, desc="", check=True, timeout=120):
    print(f"\n>>> {desc or cmd[:80]}")
    r = subprocess.run(cmd, shell=True, capture_output=True, text=True, timeout=timeout)
    out = r.stdout.strip()
    if out:
        print(out[:500])
    if check and r.returncode != 0:
        print(f"FAILED: {r.stderr[:500]}")
        raise RuntimeError(f"Command failed: {cmd}")
    return r

# ── GPU ──
r = run("nvidia-smi --query-gpu=name --format=csv,noheader", "GPU", check=False)
gpu_name = r.stdout.strip()
print(f"GPU: {gpu_name}")

# ── Load Gate 1 environment manifest ──
manifest_path = f"{GATE1_DIR}/environment.json"
if not os.path.exists(manifest_path):
    # Try Drive
    drive_manifest = None
    for sha_dir in sorted(Path(DRIVE_BASE).iterdir()) if os.path.isdir(DRIVE_BASE) else []:
        m = Path(sha_dir) / "gate1" / "environment.json"
        if m.exists():
            drive_manifest = str(m)
            break
    if drive_manifest:
        print(f"Loading manifest from Drive: {drive_manifest}")
        with open(drive_manifest) as f:
            env = json.load(f)
    else:
        raise FileNotFoundError("Gate 1 manifest not found locally or on Drive.")
else:
    with open(manifest_path) as f:
        env = json.load(f)

git_sha = env.get("accentedge_git_sha", "unknown")
print(f"accentedge SHA: {git_sha}")

# ── Path setup ──
sys.path = [p for p in sys.path if "/content" not in p]
sys.path.insert(0, FA_CODEC_DIR)
sys.path.insert(0, f"{ACCENTEDGE_DIR}/src")
os.environ["PYTHONPATH"] = FA_CODEC_DIR + "/modules:" + os.environ.get("PYTHONPATH", "")
os.chdir(FA_CODEC_DIR)

In [ ]:
# 3. Install dependencies
!pip install -q numpy soundfile librosa scipy jiwer pyyaml einops \
    huggingface-hub phonemizer speechbrain torchaudio faster-whisper \
    pytest pyworld munch plotly datasets
print("Dependencies installed")

In [ ]:
# 4. Run Gate 2 — Identity Preservation Calibration
import json, time, numpy as np, torch, librosa, types, warnings, argparse
from pathlib import Path
from datetime import datetime

warnings.simplefilter("ignore")

# ── Mock audiotools ──
def _make_mock(name):
    m = types.ModuleType(name)
    m.__path__ = []
    m.__package__ = name
    return m

mock_audio = _make_mock("audiotools")
mock_ml = _make_mock("audiotools.ml")
mock_ml.BaseModel = type("BaseModel", (), {"INTERN": [], "EXTERN": []})
mock_audio.ml = mock_ml
mock_audio.AudioSignal = type("AudioSignal", (), {})
mock_audio.STFTParams = type("STFTParams", (), {})
mock_core = _make_mock("audiotools.core")
mock_core.util = _make_mock("audiotools.core.util")
sys.modules["audiotools"] = mock_audio
sys.modules["audiotools.ml"] = mock_ml
sys.modules["audiotools.core"] = mock_core
sys.modules["audiotools.core.util"] = mock_core.util

# ── Config ���─
DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
N_PAIRS_REF = 40
N_RECON = 8
SEED = 42
TARGET_SR = 24000

class _cfg:
    facodec_dir = Path(FA_CODEC_DIR)
    facodec_ckpt = "Plachta/FAcodec"
    device = DEVICE
    n_pairs_ref = N_PAIRS_REF
    n_recon = N_RECON
    seed = SEED
    target_sr = TARGET_SR
    max_shift_over_span = 0.25
    min_preservation = 0.85

cfg = _cfg()
print(f"Device: {cfg.device}")

# ── ECAPA-TDNN ──
from speechbrain.pretrained import EncoderClassifier
print("Loading ECAPA-TDNN...")
spk_classifier = EncoderClassifier.from_hparams(
    source="speechbrain/spkrec-ecapa-voxceleb",
    savedir="/tmp/spkrec_ecapa",
    run_opts={"device": cfg.device},
)
print("ECAPA-TDNN ready")

def get_embedding(wav, sr=24000):
    if wav.ndim > 1:
        wav = wav.mean(axis=1)
    wav_t = torch.from_numpy(wav).float().unsqueeze(0).to(cfg.device)
    emb = spk_classifier.encode_batch(wav_t)
    emb = emb / emb.norm(dim=-1, keepdim=True)
    return emb.squeeze().cpu().numpy()

def cosine_sim(a, b):
    return float(np.dot(a, b))

# ── FAcodec ──
from modules.commons import build_model, recursive_munch
from hf_utils import load_custom_model_from_hf

print("Loading FAcodec...")
ckpt_path, config_path = load_custom_model_from_hf("Plachta/FAcodec")
with open(config_path) as f:
    config = yaml.safe_load(f)
mp = recursive_munch(config["model_params"])
facodec_model = build_model(mp)
ckpt = torch.load(ckpt_path, map_location="cpu")
ckpt = ckpt.get("net", ckpt)
for key in ckpt:
    facodec_model[key].load_state_dict(ckpt[key])
    facodec_model[key].eval().to(cfg.device)
    for p in facodec_model[key].parameters():
        p.requires_grad = False
print(f"FAcodec ready: {list(facodec_model.keys())}")

def facodec_reconstruct(wav_np):
    wav_t = torch.from_numpy(wav_np).float()
    if wav_t.dim() == 1:
        wav_t = wav_t.unsqueeze(0)
    wav_in = wav_t.unsqueeze(0).to(cfg.device)
    z = facodec_model["encoder"](wav_in)
    z_q, _, _, _, _ = facodec_model["quantizer"](z, wav_in, n_c=2)
    recon = facodec_model["decoder"](z_q)
    return recon.squeeze().cpu().numpy()

# ── Dataset loading ──
def _resample(wav, orig_sr, target_sr):
    return librosa.resample(np.asarray(wav, dtype=np.float32), orig_sr=orig_sr, target_sr=target_sr).astype(np.float32)

class Utt:
    __slots__ = ("speaker_id", "sentence_id", "text", "wav", "sr", "corpus")
    def __init__(self, speaker_id, sentence_id, text, wav, sr, corpus):
        self.speaker_id = speaker_id
        self.sentence_id = sentence_id
        self.text = text
        self.wav = wav
        self.sr = sr
        self.corpus = corpus

def load_cmu_arctic():
    utterances = []
    try:
        from datasets import load_dataset
        print("  Loading CMU ARCTIC from HuggingFace...")
        ds = load_dataset("cmu_arctic", split="train", trust_remote_code=True)
        speakers = sorted(set(ds["speaker_id"]))
        np.random.seed(cfg.seed)
        np.random.shuffle(speakers)
        chosen = speakers[:4]
        for spk in chosen:
            rows = [r for r in ds if r["speaker_id"] == spk]
            np.random.shuffle(rows)
            for row in rows[:cfg.n_recon + 4]:
                arr = row["audio"]["array"]
                sr = row["audio"]["sampling_rate"]
                wav = _resample(arr, sr, cfg.target_sr)
                utterances.append(Utt(spk, row.get("sentence_id", row.get("id", "")),
                    row.get("text", row.get("transcription", "")), wav, cfg.target_sr, "native"))
        print(f"  CMU ARCTIC: {len(utterances)} utterances")
        return utterances
    except Exception as e:
        print(f"  [WARN] HF CMU ARCTIC failed: {e}")

    try:
        import torchaudio
        print("  Loading CMU ARCTIC via torchaudio...")
        arctic_root = Path("/tmp/cmu_arctic")
        arctic_root.mkdir(exist_ok=True)
        speakers = ["slt", "bdl", "rms", "clb"]
        for spk in speakers:
            try:
                ds = torchaudio.datasets.CMU_ARCTIC(root=str(arctic_root), speaker=spk, download=True)
                idxs = np.random.choice(len(ds), size=min(cfg.n_recon + 4, len(ds)), replace=False)
                for idx in idxs:
                    wav, sr, transcript = ds[idx]
                    wav_np = wav.squeeze().numpy()
                    if sr != cfg.target_sr:
                        wav_np = _resample(wav_np, sr, cfg.target_sr)
                    utterances.append(Utt(spk, str(idx), transcript, wav_np, cfg.target_sr, "native"))
            except Exception as e2:
                print(f"  [WARN] Speaker {spk} failed: {e2}")
        print(f"  CMU ARCTIC (torchaudio): {len(utterances)} utterances")
        return utterances
    except Exception as e:
        print(f"  [ERROR] All CMU ARCTIC methods failed: {e}")
        return utterances

def load_l2_arctic():
    utterances = []
    try:
        from datasets import load_dataset
        print("  Loading L2-ARCTIC from HuggingFace...")
        ds = load_dataset("osCa/L2-ARCTIC", split="train", trust_remote_code=True)
        all_speakers = sorted(set(ds["speaker_id"]))
        np.random.seed(cfg.seed)
        np.random.shuffle(all_speakers)
        indian = [s for s in all_speakers if s.startswith("HI")]
        if not indian:
            print(f"  [WARN] No HI-prefixed speakers. Available: {all_speakers[:10]}")
            indian = all_speakers[:4]
        chosen = indian[:4]
        print(f"  L2-ARCTIC Indian speakers: {chosen}")
        for spk in chosen:
            rows = [r for r in ds if r["speaker_id"] == spk]
            np.random.shuffle(rows)
            for row in rows[:cfg.n_recon + 4]:
                try:
                    arr = row["audio"]["array"]
                    sr = row["audio"]["sampling_rate"]
                    wav = _resample(arr, sr, cfg.target_sr)
                    utterances.append(Utt(spk, row.get("sentence_id", row.get("id", "")),
                        row.get("transcription", row.get("text", "")), wav, cfg.target_sr, "indian"))
                except Exception as e2:
                    print(f"  [WARN] Skip {spk} row: {e2}")
        print(f"  L2-ARCTIC: {len(utterances)} utterances")
        return utterances
    except Exception as e:
        print(f"  [ERROR] L2-ARCTIC loading failed: {e}")
        return utterances

print("\n>>> Loading datasets...")
native_utts = load_cmu_arctic()
indian_utts = load_l2_arctic()
if len(native_utts) < 4:
    raise RuntimeError(f"Fewer than 4 native utterances ({len(native_utts)})")
if len(indian_utts) < 4:
    raise RuntimeError(f"Fewer than 4 Indian utterances ({len(indian_utts)})")
print(f"Native: {len(native_utts)}, Indian: {len(indian_utts)}")

In [ ]:
# 5. Build three-reference distributions
import yaml

def _summary_stats(values):
    if not values:
        return {"n": 0, "mean": 0.0, "median": 0.0, "std": 0.0, "min": 0.0, "max": 0.0}
    return {"n": len(values), "mean": float(np.mean(values)), "median": float(np.median(values)),
            "std": float(np.std(values)), "min": float(np.min(values)), "max": float(np.max(values))}

def build_speaker_groups(utterances):
    groups = {}
    for u in utterances:
        groups.setdefault(u.speaker_id, []).append(u)
    return groups

def compute_same_speaker(utterances, extractor, n_pairs=40):
    groups = build_speaker_groups(utterances)
    sims = []
    for spk, utts in groups.items():
        if len(utts) < 2:
            continue
        for i in range(len(utts)):
            for j in range(i + 1, len(utts)):
                sims.append(cosine_sim(extractor(utts[i].wav, utts[i].sr), extractor(utts[j].wav, utts[j].sr)))
                if len(sims) >= n_pairs:
                    break
            if len(sims) >= n_pairs:
                break
    return {"sims": sims, "summary": _summary_stats(sims)}

def compute_impostor(group_a, group_b, extractor, n_pairs=40):
    emb_a = [(u, extractor(u.wav, u.sr)) for u in group_a]
    emb_b = [(u, extractor(u.wav, u.sr)) for u in group_b]
    sims = []
    rng = np.random.RandomState(cfg.seed)
    for u_a, e_a in emb_a:
        candidates = [(u_b, e_b) for u_b, e_b in emb_b if u_b.speaker_id != u_a.speaker_id]
        for u_b, e_b in rng.permutation(candidates):
            sims.append(cosine_sim(e_a, e_b))
            if len(sims) >= n_pairs:
                break
        if len(sims) >= n_pairs:
            break
    return {"sims": sims, "summary": _summary_stats(sims)}

def compute_reconstruction(utts, recon_wavs, extractor):
    sims = []
    for utt, recon in zip(utts, recon_wavs):
        try:
            sims.append(cosine_sim(extractor(utt.wav, utt.sr), extractor(recon, utt.sr)))
        except Exception as e:
            print(f"  [WARN] Embedding failed: {e}")
    return {"sims": sims, "summary": _summary_stats(sims)}

print("\n>>> Building three-reference distributions...")

# 1. Same-speaker
native_ss = compute_same_speaker(native_utts, get_embedding, cfg.n_pairs_ref)
indian_ss = compute_same_speaker(indian_utts, get_embedding, cfg.n_pairs_ref)
print(f"  Native same-speaker: n={native_ss['summary']['n']} median={native_ss['summary']['median']:.4f}")
print(f"  Indian same-speaker: n={indian_ss['summary']['n']} median={indian_ss['summary']['median']:.4f}")

# 2. Impostor
native_imp = compute_impostor(native_utts, native_utts, get_embedding, cfg.n_pairs_ref)
indian_imp = compute_impostor(indian_utts, indian_utts, get_embedding, cfg.n_pairs_ref)
print(f"  Native impostor:     n={native_imp['summary']['n']} median={native_imp['summary']['median']:.4f}")
print(f"  Indian impostor:     n={indian_imp['summary']['n']} median={indian_imp['summary']['median']:.4f}")

# 3. Reconstruction
native_recon_wavs, indian_recon_wavs = [], []
for utt in native_utts[:cfg.n_recon]:
    try:
        recon = facodec_reconstruct(utt.wav)
        native_recon_wavs.append(recon)
    except Exception as e:
        print(f"  [WARN] Native recon failed: {e}")
for utt in indian_utts[:cfg.n_recon]:
    try:
        recon = facodec_reconstruct(utt.wav)
        indian_recon_wavs.append(recon)
    except Exception as e:
        print(f"  [WARN] Indian recon failed: {e}")

native_recon = compute_reconstruction(native_utts[:len(native_recon_wavs)], native_recon_wavs, get_embedding)
indian_recon = compute_reconstruction(indian_utts[:len(indian_recon_wavs)], indian_recon_wavs, get_embedding)
print(f"  Native recon:        n={native_recon['summary']['n']} median={native_recon['summary']['median']:.4f}")
print(f"  Indian recon:        n={indian_recon['summary']['n']} median={indian_recon['summary']['median']:.4f}")

# Legacy SECS diagnostic
def legacy_secs(utts, recon_wavs):
    sims = []
    for utt, recon in zip(utts, recon_wavs):
        sims.append(cosine_sim(get_embedding(utt.wav, utt.sr), get_embedding(recon, utt.sr)))
    return float(np.mean(sims)) if sims else float("nan")

print(f"\n  +{'-'*56}+")
print(f"  |  Legacy SECS mean (DIAGNOSTIC ONLY)              |")
print(f"  |  Native: {legacy_secs(native_utts[:len(native_recon_wavs)], native_recon_wavs):.4f}                                  |")
print(f"  |  Indian: {legacy_secs(indian_utts[:len(indian_recon_wavs)], indian_recon_wavs):.4f}                                  |")
print(f"  +{'-'*56}+")

In [ ]:
# 6. Calibration & Gate 2 check
def calibrate(same_spk, impostor, recon):
    ss_med = same_spk["summary"]["median"]
    imp_med = impostor["summary"]["median"]
    recon_med = recon["summary"]["median"]
    span = ss_med - imp_med
    shift = ss_med - recon_med
    shift_over_span = shift / span if span > 1e-9 else float("nan")
    preservation = recon_med / ss_med if ss_med > 1e-9 else float("nan")
    return {"same_speaker_median": float(ss_med), "impostor_median": float(imp_med),
            "recon_median": float(recon_med), "same_speaker_span": float(span),
            "reconstruction_shift": float(shift), "shift_over_span": float(shift_over_span),
            "preservation_ratio": float(preservation)}

def check_gate2(cal_native, cal_indian):
    results = {}
    for corpus, cal in [("native", cal_native), ("indian", cal_indian)]:
        sos = cal["shift_over_span"]
        pres = cal["preservation_ratio"]
        shift_ok = not np.isnan(sos) and sos < cfg.max_shift_over_span
        pres_ok = not np.isnan(pres) and pres > cfg.min_preservation
        results[corpus] = {"shift_over_span": float(sos) if not np.isnan(sos) else None,
            "preservation_ratio": float(pres) if not np.isnan(pres) else None,
            "shift_over_span_pass": bool(shift_ok), "preservation_ratio_pass": bool(pres_ok),
            "gate2_pass": bool(shift_ok and pres_ok)}
    overall = results["native"]["gate2_pass"] and results["indian"]["gate2_pass"]
    return results, overall

cal_native = calibrate(native_ss, native_imp, native_recon)
cal_indian = calibrate(indian_ss, indian_imp, indian_recon)
gate, overall_pass = check_gate2(cal_native, cal_indian)

print("\n" + "="*64)
print("  GATE 2 — IDENTITY PRESERVATION CALIBRATION")
print("="*64)
def fmt(v):
    if v is None:
        return "N/A"
    return f"{v:.4f}"
print(f"  {'Metric':<30} {'Native':>12} {'Indian':>12} {'Threshold':>10}")
print(f"  {'-'*64}")
print(f"  {'same-speaker median':<30} {fmt(cal_native['same_speaker_median']):>12} {fmt(cal_indian['same_speaker_median']):>12}")
print(f"  {'impostor median':<30} {fmt(cal_native['impostor_median']):>12} {fmt(cal_indian['impostor_median']):>12}")
print(f"  {'recon median':<30} {fmt(cal_native['recon_median']):>12} {fmt(cal_indian['recon_median']):>12}")
print(f"  {'same-speaker span':<30} {fmt(cal_native['same_speaker_span']):>12} {fmt(cal_indian['same_speaker_span']):>12}")
print(f"  {'reconstruction shift':<30} {fmt(cal_native['reconstruction_shift']):>12} {fmt(cal_indian['reconstruction_shift']):>12}")
print(f"  {'shift / span':<30} {fmt(cal_native['shift_over_span']):>12} {fmt(cal_indian['shift_over_span']):>12} "
      f"< {cfg.max_shift_over_span}   {'PASS' if gate['native']['shift_over_span_pass'] else 'FAIL'} / {'PASS' if gate['indian']['shift_over_span_pass'] else 'FAIL'}")
print(f"  {'preservation ratio':<30} {fmt(cal_native['preservation_ratio']):>12} {fmt(cal_indian['preservation_ratio']):>12} "
      f"> {cfg.min_preservation}   {'PASS' if gate['native']['preservation_ratio_pass'] else 'FAIL'} / {'PASS' if gate['indian']['preservation_ratio_pass'] else 'FAIL'}")
print(f"  {'-'*64}")
print(f"  {'GATE 2 PASS':<30} {'YES' if gate['native']['gate2_pass'] else 'NO':>12} {'YES' if gate['indian']['gate2_pass'] else 'NO':>12}")
print(f"  {'OVERALL':<30} {'':>12} {'':>12} {'PASS' if overall_pass else 'FAIL':>10}")
print("="*64)

In [ ]:
# 7. Save results
os.makedirs(GATE2_DIR, exist_ok=True)
drive_out = f"{DRIVE_BASE}/{git_sha}/gate2"
os.makedirs(drive_out, exist_ok=True)

def _safe(v):
    if isinstance(v, float) and (np.isnan(v) or np.isinf(v)):
        return None
    return v

output = {
    "gate": 2, "gate_name": "Identity Preservation Calibration",
    "timestamp": datetime.now().isoformat(),
    "config": {"device": cfg.device, "seed": cfg.seed, "facodec_ckpt": cfg.facodec_ckpt,
               "n_pairs_ref": cfg.n_pairs_ref, "n_recon": cfg.n_recon,
               "max_shift_over_span": cfg.max_shift_over_span, "min_preservation": cfg.min_preservation},
    "native": {"calibration": {k: _safe(v) for k, v in cal_native.items()},
               "reference_distributions": {
                   "same_speaker": {"n": native_ss["summary"]["n"], "median": _safe(native_ss["summary"]["median"]), "mean": _safe(native_ss["summary"]["mean"])},
                   "impostor": {"n": native_imp["summary"]["n"], "median": _safe(native_imp["summary"]["median"]), "mean": _safe(native_imp["summary"]["mean"])},
                   "reconstruction": {"n": native_recon["summary"]["n"], "median": _safe(native_recon["summary"]["median"]), "mean": _safe(native_recon["summary"]["mean"])}},
               "gate_result": gate["native"]},
    "indian": {"calibration": {k: _safe(v) for k, v in cal_indian.items()},
               "reference_distributions": {
                   "same_speaker": {"n": indian_ss["summary"]["n"], "median": _safe(indian_ss["summary"]["median"]), "mean": _safe(indian_ss["summary"]["mean"])},
                   "impostor": {"n": indian_imp["summary"]["n"], "median": _safe(indian_imp["summary"]["median"]), "mean": _safe(indian_imp["summary"]["mean"])},
                   "reconstruction": {"n": indian_recon["summary"]["n"], "median": _safe(indian_recon["summary"]["median"]), "mean": _safe(indian_recon["summary"]["mean"])}},
               "gate_result": gate["indian"]},
    "overall": {"gate2_pass": bool(overall_pass)},
}

# Save locally
with open(f"{GATE2_DIR}/identity_calibration.json", "w") as f:
    json.dump(output, f, indent=2)
print(f"Saved: {GATE2_DIR}/identity_calibration.json")

# Save to Drive
for fname in ["identity_calibration.json"]:
    src = f"{GATE2_DIR}/{fname}"
    if os.path.exists(src):
        shutil.copy2(src, f"{drive_out}/{fname}")
print(f"Drive: {drive_out}")

if not overall_pass:
    raise RuntimeError("GATE 2 FAILED — aborting pipeline.")

In [ ]:
# 8. Gate 2 complete
print(f"\nGate 2 {'PASSED' if overall_pass else 'FAILED'}.")
print(f"Results: {GATE2_DIR}/identity_calibration.json")